# Dimenzionisanje tranzistora LDO-a preko gm/ID LUT baze

Sveska prima sledeće projektne ulaze:

- $L_d$ i $L_{pass}$ u nm;
- ciljne vrednosti $(g_m/I_D)$ za pass, n1, p1, n2 i p2 u 1/V;
- struju pass grane;
- struju jedne grane prvog stepena, zajedničku za n1 i p1;
- struju drugog stepena, zajedničku za n2 i p2.

Za svaki tranzistor računa se stvarna radna tačka, gustina struje

$$
J_D=\frac{I_D}{W},
$$

a zatim ukupna potrebna širina

$$
W=\frac{I_D}{J_D}.
$$

Za PMOS se koriste pozitivne magnitude $V_{SG}$, $V_{SD}$ i $V_{BS}$.
Dobijeni $W$ predstavlja ukupnu efektivnu širinu; broj prstiju i širina
jednog prsta biraju se naknadno u layout-u.

Kapacitivnosti `CGG`, `CGS`, `CSG`, `CGD`, `CDG`, `CGB`, `CDD` i `CSS` interpoliraju se u istoj radnoj tački i skaliraju na izračunatu ukupnu širinu $W$.


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import brentq

from gmid_toolbox import GMIDToolbox, LUTError

pd.set_option("display.max_columns", 100)
np.set_printoptions(precision=6, suppress=True)

DATA_DIR = Path.cwd()

tb = GMIDToolbox(
    nmos_mat=DATA_DIR / "65nch.mat",
    pmos_mat=DATA_DIR / "65pch.mat",
    nmos_variable="nch",
    pmos_variable="pch",
)

LUT1, LUT2, LUT3, LUT4 = tb.LUT1, tb.LUT2, tb.LUT3, tb.LUT4
print(tb.info())


{'nmos': {'kind': 'nmos', 'L_um': (0.06, 1.0), 'VGS_or_VSG_V': (0.0, 1.2), 'VDS_or_VSD_V': (0.0, 1.2), 'VSB_or_VBS_V': (0.0, 0.8)}, 'pmos': {'kind': 'pmos', 'L_um': (0.06, 1.0), 'VGS_or_VSG_V': (0.0, 1.2), 'VDS_or_VSD_V': (0.0, 1.2), 'VSB_or_VBS_V': (0.0, 0.8)}}


## 1. Projektni ulazi

Struje se unose u amperima. Na primer, `20e-6` predstavlja 20 µA.
`I_STAGE1_BRANCH_A` je struja jednog n1/p1 para, a ne zbir obe grane
diferencijalnog para.


In [12]:
# Glavni ulazi koje menjaš
LD_NM = 130.0
LPASS_NM = 65.0

GMID_PASS = 7.0
GMID_P2 = 18.0
GMID_N1 = 18.0
GMID_P1 = 18.0
GMID_N2 = 18.0

I_PASS_A = 5e-3
I_STAGE1_BRANCH_A = 20e-6
I_STAGE2_A = 100e-6

# Fiksni naponi šeme
VDD = 1.20
VREF = 0.50
VOUT = 1.00
VSD_PASS = 0.10

# Body bias, kao u prethodnom proračunu
VSB_N = 0.0
VBS_P = 0.0


## 2. DC solver i funkcija za dimenzionisanje

Funkcija `size_ldo(...)` najpre rešava $V_{G0}$, $V_{G6}$, $V_S$,
$V_{DS3}$, zatim $V_{D3}=V_S+V_{DS3}$ i $V_{D5}$. Zatim LUT direktno vraća `gm/ID`,
`gds/ID` i $J_D$ u svakoj stvarnoj radnoj tački.


In [13]:
ROOT_EPS_V = 1e-7
ROOT_TOL_V = 1e-9


def _all_roots(func, low, high, *, n_scan=121):
    '''Nalazi sve validne promene znaka bez LUT ekstrapolacije.'''
    samples = []
    for x in np.linspace(float(low), float(high), int(n_scan)):
        try:
            y = float(func(float(x)))
        except (LUTError, ValueError, FloatingPointError):
            continue
        if np.isfinite(y):
            samples.append((float(x), y))

    roots = []
    for x, y in samples:
        if abs(y) <= 1e-10:
            roots.append(x)

    for (x0, y0), (x1, y1) in zip(samples[:-1], samples[1:]):
        if y0 * y1 < 0:
            roots.append(float(brentq(func, x0, x1, xtol=ROOT_TOL_V)))

    return sorted(set(np.round(roots, 12)))


def _solve_vs(*, gmid_n1, VG6, Ld_nm, Vref, vsb_n):
    info = tb.info()["nmos"]
    vgs_min, vgs_max = info["VGS_or_VSG_V"]
    vds_min, vds_max = info["VDS_or_VSD_V"]

    low = max(Vref - vgs_max, VG6 - vds_max) + ROOT_EPS_V
    high = min(Vref - vgs_min, VG6 - vds_min) - ROOT_EPS_V
    if low >= high:
        raise LUTError("Ne postoji validan VS interval unutar LUT opsega.")

    def equation(VS):
        return LUT3(
            "nch",
            Vref - VS,
            VG6 - VS,
            vsb_n,
            Ld_nm,
        ) - float(gmid_n1)

    try:
        f_low, f_high = equation(low), equation(high)
        if f_low == 0:
            return float(low)
        if f_high == 0:
            return float(high)
        if f_low * f_high < 0:
            return float(brentq(equation, low, high, xtol=ROOT_TOL_V))
    except (LUTError, ValueError, FloatingPointError):
        pass

    roots = _all_roots(equation, low, high)
    if not roots:
        raise LUTError(
            f"gm/ID_n1={gmid_n1:g} 1/V nije ostvariv za VG6={VG6:.5f} V."
        )
    return float(min(roots, key=abs))


def _solve_vd5(*, VD3, VG0, Vdd, Ld_nm, vsb_n):
    def vd5_map(VD5):
        gmid_n2 = LUT3("nch", VD5, VG0, vsb_n, Ld_nm)
        return Vdd - LUT1(
            "nch", gmid_n2, Vdd - VD3, vsb_n, Ld_nm
        )

    vd5 = 0.5 * Vdd
    for _ in range(40):
        mapped = float(vd5_map(vd5))
        if abs(mapped - vd5) <= ROOT_TOL_V:
            return mapped
        vd5 = 0.5 * vd5 + 0.5 * mapped

    info = tb.info()["nmos"]
    vgs_min, vgs_max = info["VGS_or_VSG_V"]
    low = max(vgs_min, 0.0) + ROOT_EPS_V
    high = min(vgs_max, Vdd) - ROOT_EPS_V
    equation = lambda value: value - vd5_map(value)
    roots = _all_roots(equation, low, high, n_scan=201)
    if not roots:
        raise LUTError("Implicitna jednačina za VD5 nema validno rešenje.")
    return float(min(roots, key=lambda value: abs(value - vd5)))


def size_ldo(
    *,
    Ld_nm,
    Lpass_nm,
    gmid_pass,
    gmid_p2,
    gmid_n1,
    gmid_p1,
    gmid_n2,
    Ipass_A,
    Istage1_branch_A,
    Istage2_A,
    Vdd=1.20,
    Vref=0.50,
    Vout=1.00,
    Vsd_pass=0.10,
    Vsb_n=0.0,
    Vbs_p=0.0,
):
    '''Vraća (tabelu tranzistora, tabelu čvorova) za zadate ulaze.'''
    currents = {
        "pass": float(Ipass_A),
        "n1": float(Istage1_branch_A),
        "p1": float(Istage1_branch_A),
        "n2": float(Istage2_A),
        "p2": float(Istage2_A),
    }
    if any(current <= 0 for current in currents.values()):
        raise ValueError("Sve granske struje moraju biti pozitivne.")

    # Zavisni DC čvorovi
    VSG_pass = LUT1("pch", gmid_pass, Vsd_pass, Vbs_p, Lpass_nm)
    VG0 = Vdd - VSG_pass

    VG6 = Vdd - LUT1(
        "pch", gmid_p2, Vdd - VG0, Vbs_p, Ld_nm
    )
    VS = _solve_vs(
        gmid_n1=gmid_n1,
        VG6=VG6,
        Ld_nm=Ld_nm,
        Vref=Vref,
        vsb_n=Vsb_n,
    )

    expected_vds_n1 = VG6 - VS
    VDS3 = LUT2(
        "nch",
        gmid_n1,
        Vref - VS,
        Vsb_n,
        Ld_nm,
        preferred_vds=expected_vds_n1,
        relative_tolerance=1e-5,
        n_scan=161,
    )
    # LUT2 vraća terminalni VDS3; napon čvora je VD3=VS+VDS3.
    VD3 = VS + VDS3
    # gmIDn2 je sada nezavisan korisnički ulaz, prema zadatoj relaciji.
    VD5 = Vdd - LUT1(
        "nch", gmid_n2, Vdd - VD3, Vsb_n, Ld_nm
    )

    bias = {
        "pass": {
            "mos": "pch", "L_nm": Lpass_nm,
            "gmid_target": gmid_pass,
            "VGS_V": VSG_pass, "VDS_V": Vsd_pass, "VSB_V": Vbs_p,
        },
        "n1": {
            "mos": "nch", "L_nm": Ld_nm,
            "gmid_target": gmid_n1,
            "VGS_V": Vref - VS, "VDS_V": VDS3, "VSB_V": Vsb_n,
        },
        "p1": {
            "mos": "pch", "L_nm": Ld_nm,
            "gmid_target": gmid_p1,
            "VGS_V": Vdd - VG6, "VDS_V": VDS3, "VSB_V": Vbs_p,
        },
        "n2": {
            "mos": "nch", "L_nm": Ld_nm,
            "gmid_target": gmid_n2,
            "VGS_V": VD5, "VDS_V": VG0, "VSB_V": Vsb_n,
        },
        "p2": {
            "mos": "pch", "L_nm": Ld_nm,
            "gmid_target": gmid_p2,
            "VGS_V": Vdd - VG6, "VDS_V": Vdd - VG0, "VSB_V": Vbs_p,
        },
    }

    rows = []
    for name, point in bias.items():
        gmid_actual = LUT3(
            point["mos"], point["VGS_V"], point["VDS_V"],
            point["VSB_V"], point["L_nm"],
        )
        gmid_target = float(point["gmid_target"])
        required_vgs = LUT1(
            point["mos"], gmid_target, point["VDS_V"],
            point["VSB_V"], point["L_nm"],
        )
        gdsid = tb.gds_id_at_bias(
            mos=point["mos"],
            vgs=required_vgs,
            vds=point["VDS_V"],
            vsb=point["VSB_V"],
            L_nm=point["L_nm"],
        )
        jd = tb.jd_at_bias(
            mos=point["mos"],
            vgs=required_vgs,
            vds=point["VDS_V"],
            vsb=point["VSB_V"],
            L_nm=point["L_nm"],
        )
        if not np.isfinite(jd) or jd <= 0:
            raise LUTError(f"{name}: JD nije pozitivna konačna vrednost.")

        current = currents[name]
        width_um = current / jd
        caps_lut = tb.capacitances_at_bias(
            mos=point["mos"], vgs=required_vgs, vds=point["VDS_V"],
            vsb=point["VSB_V"], L_nm=point["L_nm"],
        )
        cap_scale = width_um / caps_lut["W_REF_UM"]
        caps_ff = {
            cap_name: caps_lut[cap_name] * cap_scale * 1e15
            for cap_name in ("CGG", "CGS", "CSG", "CGD", "CDG", "CGB", "CDD", "CSS")
        }
        gm_S = gmid_target * current
        gds_S = gdsid * current

        rows.append({
            "tranzistor": name,
            "tip": point["mos"],
            "L [nm]": point["L_nm"],
            "ID [A]": current,
            "ID [uA]": current * 1e6,
            "VGS/VSG [V]": point["VGS_V"],
            "VDS/VSD [V]": point["VDS_V"],
            "zadati gm/ID [1/V]": gmid_target,
            "stvarni gm/ID [1/V]": gmid_actual,
            "delta gm/ID [1/V]": gmid_actual - gmid_target,
            "gds/ID [1/V]": gdsid,
            "potreban VGS/VSG [V]": required_vgs,
            "delta VGS/VSG [mV]": (point["VGS_V"] - required_vgs) * 1e3,
            "JD [A/um]": jd,
            "JD [uA/um]": jd * 1e6,
            "W [um]": width_um,
            "W/L": width_um / (point["L_nm"] / 1000.0),
            "gm [uS]": gm_S * 1e6,
            "gds [uS]": gds_S * 1e6,
            "ro [kohm]": 1.0 / gds_S / 1e3,
            **{f"{cap_name} [fF]": value for cap_name, value in caps_ff.items()},
        })

    transistor_table = pd.DataFrame(rows).set_index("tranzistor")
    node_table = pd.DataFrame({
        "napon [V]": {
            "VDD": Vdd,
            "VOUT": Vout,
            "VG0": VG0,
            "VG6": VG6,
            "VS": VS,
            "VDS3": VDS3,
            "VD3 = VS + VDS3": VD3,
            "VD5": VD5,
        }
    })
    return transistor_table, node_table


## 3. Proračun širina

Promeni vrednosti u ulaznoj ćeliji i ponovo pokreni ovu ćeliju.


In [14]:
widths, node_voltages = size_ldo(
    Ld_nm=LD_NM,
    Lpass_nm=LPASS_NM,
    gmid_pass=GMID_PASS,
    gmid_p2=GMID_P2,
    gmid_n1=GMID_N1,
    gmid_p1=GMID_P1,
    gmid_n2=GMID_N2,
    Ipass_A=I_PASS_A,
    Istage1_branch_A=I_STAGE1_BRANCH_A,
    Istage2_A=I_STAGE2_A,
    Vdd=VDD,
    Vref=VREF,
    Vout=VOUT,
    Vsd_pass=VSD_PASS,
    Vsb_n=VSB_N,
    Vbs_p=VBS_P,
)

print("Rešeni DC čvorovi:")
display(node_voltages.round(6))

print("Dimenzije tranzistora:")
display(widths.round({
    "VGS/VSG [V]": 6,
    "VDS/VSD [V]": 6,
    "zadati gm/ID [1/V]": 5,
    "stvarni gm/ID [1/V]": 5,
    "delta gm/ID [1/V]": 5,
    "gds/ID [1/V]": 5,
    "potreban VGS/VSG [V]": 6,
    "delta VGS/VSG [mV]": 3,
    "JD [uA/um]": 5,
    "W [um]": 5,
    "W/L": 3,
    "gm [uS]": 3,
    "gds [uS]": 3,
    "ro [kohm]": 3,
    "CGG [fF]": 3, "CGS [fF]": 3, "CSG [fF]": 3,
    "CGD [fF]": 3, "CDG [fF]": 3, "CGB [fF]": 3,
    "CDD [fF]": 3, "CSS [fF]": 3,
}))

print("Kapacitivnosti tranzistora, skalirane na izračunato W:")
capacitances = widths[[
    "CGG [fF]", "CGS [fF]", "CSG [fF]", "CGD [fF]",
    "CDG [fF]", "CGB [fF]", "CDD [fF]", "CSS [fF]",
]].copy()
display(capacitances.round(3))


Rešeni DC čvorovi:


,napon [V]
VDD,1.200000
VOUT,1.000000
VG0,0.493454
VG6,0.706660
VS,0.041667
VDS3,0.664994
VD3 = VS + VDS3,0.706660
VD5,0.738858


Dimenzije tranzistora:


,tip,L [nm],ID [A],ID [uA],VGS/VSG [V],VDS/VSD [V],zadati gm/ID [1/V],stvarni gm/ID [1/V],delta gm/ID [1/V],gds/ID [1/V],potreban VGS/VSG [V],delta VGS/VSG [mV],JD [A/um],JD [uA/um],W [um],W/L,gm [uS],gds [uS],ro [kohm],CGG [fF],CGS [fF],CSG [fF],CGD [fF],CDG [fF],CGB [fF],CDD [fF],CSS [fF]
tranzistor,,,,,,,,,,,,,,,,,,,,,,,,,,,
pass,pch,65.0,0.00500,5000.0,0.706546,0.100000,7.0,7.00000,0.00000,6.16470,0.706546,0.000,0.000015,15.00422,333.23955,5126.762,35000.0,30823.494,0.032,394.756,226.553,193.356,157.708,187.637,10.495,264.431,315.657
n1,nch,130.0,0.00002,20.0,0.458333,0.664994,18.0,18.00000,0.00000,0.54185,0.458333,0.000,0.000004,3.93561,5.08181,39.091,360.0,10.837,92.277,6.576,4.185,3.306,1.698,2.495,0.693,3.164,5.447
p1,pch,130.0,0.00002,20.0,0.493340,0.664994,18.0,18.04840,0.04840,0.59349,0.493989,-0.649,0.000002,2.12198,9.42515,72.501,360.0,11.870,84.247,11.914,7.290,5.928,3.214,4.494,1.410,5.545,9.815
n2,nch,130.0,0.00010,100.0,0.738858,0.493454,18.0,6.10386,-11.89614,0.63980,0.461140,277.718,0.000004,3.74963,26.66929,205.148,1800.0,63.980,15.630,34.967,22.289,17.479,9.097,13.465,3.581,17.207,28.728
p2,pch,130.0,0.00010,100.0,0.493340,0.706546,18.0,18.00000,-0.00000,0.57249,0.493340,0.000,0.000002,2.15075,46.49533,357.656,1800.0,57.249,17.468,58.555,35.846,29.194,15.730,21.982,6.979,27.030,48.364


Kapacitivnosti tranzistora, skalirane na izračunato W:


,CGG [fF],CGS [fF],CSG [fF],CGD [fF],CDG [fF],CGB [fF],CDD [fF],CSS [fF]
tranzistor,,,,,,,,
pass,394.756,226.553,193.356,157.708,187.637,10.495,264.431,315.657
n1,6.576,4.185,3.306,1.698,2.495,0.693,3.164,5.447
p1,11.914,7.290,5.928,3.214,4.494,1.410,5.545,9.815
n2,34.967,22.289,17.479,9.097,13.465,3.581,17.207,28.728
p2,58.555,35.846,29.194,15.730,21.982,6.979,27.030,48.364


## 4. Sažeta tabela za unos u šemu

Ako tranzistor deliš na `NF` jednakih prstiju, približna širina jednog
prsta je `W/NF`. Izabrani `NF` mora biti ceo broj i treba ga proveriti
pravilima layout-a konkretne 65 nm tehnologije.


In [15]:
schematic_sizes = widths[[
    "tip", "L [nm]", "ID [uA]", "zadati gm/ID [1/V]",
    "stvarni gm/ID [1/V]", "delta VGS/VSG [mV]",
    "JD [uA/um]", "W [um]", "W/L",
]].copy()

display(schematic_sizes.round(5))

# Po potrebi sačuvaj tabelu:
# schematic_sizes.to_csv("LDO_transistor_widths.csv")


,tip,L [nm],ID [uA],zadati gm/ID [1/V],stvarni gm/ID [1/V],delta VGS/VSG [mV],JD [uA/um],W [um],W/L
tranzistor,,,,,,,,,
pass,pch,65.0,5000.0,7.0,7.00000,0.00000,15.00422,333.23955,5126.76229
n1,nch,130.0,20.0,18.0,18.00000,0.00000,3.93561,5.08181,39.09082
p1,pch,130.0,20.0,18.0,18.04840,-0.64939,2.12198,9.42515,72.50112
n2,nch,130.0,100.0,18.0,6.10386,277.71752,3.74963,26.66929,205.14838
p2,pch,130.0,100.0,18.0,18.00000,0.00000,2.15075,46.49533,357.65636


## Važne napomene

- Širine su određene iz DC karakterizacije $W=I_D/J_D$.
- Kapacitivnosti su interpolirane iz LUT baze i linearno skalirane sa $W$; fazna margina i odziv na promenu opterećenja nisu automatski optimizovani.
- Veći $g_m/I_D$ obično daje manji $J_D$, pa je za istu struju potreban
  veći $W$.
- Zbog povezanih gejtova svih pet ciljnih $g_m/I_D$ vrednosti nisu
  međusobno nezavisne. Kolone `stvarni gm/ID` i `delta VGS/VSG`
  predstavljaju proveru konzistentnosti unetih ciljeva sa DC čvorovima.
- Ako n1 i p1, odnosno n2 i p2, nemaju jednake struje u stvarnoj šemi,
  funkciju treba proširiti zasebnim strujama za svaki tranzistor.
- Tačke sa negativnim $V_S$ mogu biti unutar LUT baze, ali nisu fizički
  ostvarive kada je $V_{SS}=0$ i nema negativnog napajanja.
